# H infinity condition for stabilization of a continuous time system

In [1]:
import numpy as np
import cvxpy as cp
import control as ct

Matrizes do sistema

In [61]:
A  = np.array([
    [0, 1],
    [0, 0]
])
Bu = np.array([
    [0],
    [5*9.81/7]
])
Bw = np.array([
    [0.1],
    [0.1]
])
C  = np.array([
    [1, 0]
])
# C = np.eye(A.shape[0])
Du = np.zeros(shape=(C.shape[0], Bu.shape[1]))
Dw = np.zeros(shape=(C.shape[0], Bw.shape[1]))

In [ ]:
nx = A.shape[0]
nu = Bu.shape[1]
nc = C.shape[0]

eps = 10e-19 # 

gamma = cp.Variable()
Z = cp.Variable((nx, nx), symmetric=True)
W = cp.Variable((nu, nx))
J = cp.Variable((nx, nx), symmetric=True)

constrains = []
constrains += [ J >> eps ]

B11 = -A@Z - Bu@W - Z@A.T - W.T@Bu.T
B12 = J + Z - Z@A.T - W.T@Bu.T
B13 = Z@C.T + W.T@Du.T
B14 = -Bw

B22 = 2*Z
B23 = np.zeros(shape=(nx, nc))
B24 = -Bw

B33 = -np.eye(nc, dtype=float)
B34 = +Dw

B44 = -np.eye(1)*gamma

block = cp.bmat([
    [B11  , B12  , B13  , B14],
    [B12.T, B22  , B23  , B24],
    [B13.T, B23.T, B33  , B34],
    [B14.T, B24.T, B34.T, B44]
])
constrains += [ block << -eps]

prob = cp.Problem(cp.Minimize(gamma), constraints=constrains)
prob.solve(solver=cp.MOSEK, verbose=True)

                                     CVXPY                                     
                                     v1.5.3                                    
(CVXPY) Dec 09 06:54:14 PM: Your problem has 11 variables, 40 constraints, and 0 parameters.
(CVXPY) Dec 09 06:54:14 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Dec 09 06:54:14 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Dec 09 06:54:14 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Dec 09 06:54:14 PM: Your problem is compiled with the CPP canonicalization backend.
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
(CVXPY) Dec 09 06:54:14 PM: Compiling problem (target solver=MOSEK).
(C

0.0001273612948626655

In [81]:
X = np.linalg.inv(Z.value)
K = W.value@X
K

array([[-1159.88870499,   -30.88523056]])

In [83]:
P = X@J.value@X
P

array([[1.85136794e+00, 2.47038456e-02],
       [2.47038456e-02, 4.51601921e-04]])

In [ ]:
all(np.linalg.eig(P).eigenvalues > 0)

True

: 

In [79]:
gamma.value

array(0.00012736)